## Q3: Dot Product (MAC) of two vectors hA & hB, put result in variable hProd 

In [1]:
%%writefile q3.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N          60
#define LOW        10
#define HIGH       99
#define BLOCK_SIZE  8

// ── CUDA Kernel 
// Each thread computes one product, stored in partial array
__global__ void dotProductKernel(int *dA, int *dB, long long *dProd, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dProd[i] = (long long)dA[i] * dB[i];   /* MAC: each thread multiplies */
}

// ── Host Function 
__host__ long long dotProduct(int *hA, int *hB, int n)
{
    int       *dA, *dB;
    long long *dProd;
    long long *hProdArr;
    long long  hProd = 0;
    int   sizeInt  = n * sizeof(int);
    int   sizeProd = n * sizeof(long long);

    hProdArr = (long long*) malloc(sizeProd);

    cudaMalloc((void**)&dA,    sizeInt);
    cudaMalloc((void**)&dB,    sizeInt);
    cudaMalloc((void**)&dProd, sizeProd);

    cudaMemcpy(dA, hA, sizeInt, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, sizeInt, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    dotProductKernel<<<DimGrid, DimBlock>>>(dA, dB, dProd, n);

    cudaMemcpy(hProdArr, dProd, sizeProd, cudaMemcpyDeviceToHost);

    // Final sum on host
    for(int i=0; i<n; i++)
        hProd += hProdArr[i];

    cudaFree(dA); cudaFree(dB); cudaFree(dProd);
    free(hProdArr);

    return hProd;
}

// ── Main 
int main()
{
    int       *hA, *hB;
    long long  hProd;

    hA = (int*) malloc(N * sizeof(int));
    hB = (int*) malloc(N * sizeof(int));

    srand(time(NULL));
    for(int i=0; i<N; i++)
    {
        hA[i] = rand()%90+10;
        hB[i] = rand()%90+10;
    }

    printf("hA: "); for(int i=0;i<N;i++) printf("%4d",hA[i]); printf("\n");
    printf("hB: "); for(int i=0;i<N;i++) printf("%4d",hB[i]); printf("\n");

    hProd = dotProduct(hA, hB, N);
    printf("\nDot Product (hProd) = %lld\n", hProd);

    free(hA); free(hB);
    return 0;
}

Writing q3.cu


In [2]:
!nvcc -arch=sm_75 q3.cu -o q3

!./q3

hA:   64  87  48  78  57  29  80  91  92  12  72  87  88  10  58  50  25  62  32  61  14  68  69  60  57  18  76  41  52  22  16  75  51  99  81  16  41  31  47  38  19  38  55  64  11  61  90  15  26  67  65  97  23  73  24  69  99  71  29  14
hB:   88  87  43  46  22  88  88  76  45  42  62  60  95  95  74  22  37  63  92  80  81  84  21  23  24  29  95  26  42  11  62  83  89  36  60  47  36  62  43  56  98  47  69  98  76  70  36  41  14  59  35  58  80  23  11  33  97  16  87  82

Dot Product (hProd) = 184420
